# Unidad 2 - Algoritmos de Regresion (Clase en Vivo)

**Dataset:** California Housing (version cruda con variable categorica)

**Objetivos de la clase:**
1. Entender que el escalado NO afecta a la regresion lineal ordinaria.
2. Ver el impacto del escalado en modelos con regularizacion (Ridge).
3. Visualizar el overfitting en Arboles de Decision controlando `max_depth`.
4. Aplicar imputacion de nulos y One-Hot Encoding.
5. Evaluar modelos con MAE, RMSE, R2 y Cross-Validation.
6. Introducir Feature Engineering con ayuda de IA.

## 1. Importacion de librerias

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

%matplotlib inline

print("[OK] Librerias cargadas correctamente.")

## 2. Carga del dataset (California Housing)

Descargamos el dataset crudo que contiene la variable categorica "ocean_proximity".

In [ ]:
url = "housing.csv"
df = pd.read_csv(url)

print("[INFO] Vista rapida del dataset:")
display(df.head())

print(f"\n[INFO] Dimensiones del dataset: {df.shape}")
print("\n[INFO] Tipos de datos y valores nulos:")
display(df.info())

## 3. Manejo de valores nulos (Imputacion)

Imputamos `total_bedrooms` con la mediana (mas robusta que la media).

In [ ]:
print("[INFO] Valores nulos por columna:")
print(df.isnull().sum())

imputer = SimpleImputer(strategy='median')
df['total_bedrooms'] = imputer.fit_transform(df[['total_bedrooms']])

print("\n[OK] Verificamos que ya no hay nulos en 'total_bedrooms':")
print(df.isnull().sum())

## 4. One-Hot Encoding (Variable categorica)

`ocean_proximity` es nominal -> One-Hot con `drop_first=True`.

In [ ]:
df_encoded = pd.get_dummies(df, columns=['ocean_proximity'], drop_first=True)

print("[INFO] Columnas despues del One-Hot Encoding:")
display(df_encoded.head())

## 5. Division en Features (X) y Target (y)

In [ ]:
X = df_encoded.drop('median_house_value', axis=1)
y = df_encoded['median_house_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"[INFO] Tamano de entrenamiento: {X_train.shape[0]} filas")
print(f"[INFO] Tamano de prueba: {X_test.shape[0]} filas")

## Experimento 1: Regresion Lineal SIN escalar

In [ ]:
print("\n" + "="*80)
print("EXPERIMENTO 1: REGRESION LINEAL SIN ESCALAR")
print("="*80)

lr_unscaled = LinearRegression()
lr_unscaled.fit(X_train, y_train)
y_pred_lr_unscaled = lr_unscaled.predict(X_test)

rmse_lr_unscaled = np.sqrt(mean_squared_error(y_test, y_pred_lr_unscaled))
r2_lr_unscaled = r2_score(y_test, y_pred_lr_unscaled)

print(f"RMSE sin escalar: ${rmse_lr_unscaled:,.2f}")
print(f"R2 sin escalar: {r2_lr_unscaled:.4f}")

## Experimento 2: Regresion Lineal CON escalado (StandardScaler)

**Observacion:** El RMSE y R2 son IDENTICOS a los del experimento 1.
Esto se debe a que la regresion lineal ordinaria (sin regularizacion) es invariante a la escala de las features.
Los coeficientes se ajustan automaticamente para compensar las diferencias de escala.

**Conclusion:** El escalado NO es necesario para LinearRegression. Pero SI lo es para modelos con regularizacion (Ridge, Lasso), SVR, k-NN, etc.

In [ ]:
print("\n" + "="*80)
print("EXPERIMENTO 2: REGRESION LINEAL CON ESCALADO (StandardScaler)")
print("="*80)

pipeline_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression())
])

pipeline_lr.fit(X_train, y_train)
y_pred_lr_scaled = pipeline_lr.predict(X_test)

rmse_lr_scaled = np.sqrt(mean_squared_error(y_test, y_pred_lr_scaled))
r2_lr_scaled = r2_score(y_test, y_pred_lr_scaled)

print(f"RMSE con escalado: ${rmse_lr_scaled:,.2f}")
print(f"R2 con escalado: {r2_lr_scaled:.4f}")
print(f"Diferencia de RMSE: ${rmse_lr_unscaled - rmse_lr_scaled:,.2f} (praticamente cero)")
print("[CONCLUSION] El escalado NO mejora la regresion lineal ordinaria.")

## Experimento 3: Ridge Regression (con regularizacion) - El escalado SÍ importa

Ridge penaliza los coeficientes. Si las features tienen escalas muy diferentes, la penalizacion afecta desproporcionadamente a las de menor escala.
Escalar las features garantiza que la regularizacion actue de forma uniforme.

In [ ]:
print("\n" + "="*80)
print("EXPERIMENTO 3: RIDGE REGRESSION - IMPACTO DEL ESCALADO")
print("="*80)

# Ridge sin escalar
ridge_unscaled = Ridge(alpha=1.0)
ridge_unscaled.fit(X_train, y_train)
y_pred_ridge_unscaled = ridge_unscaled.predict(X_test)
rmse_ridge_unscaled = np.sqrt(mean_squared_error(y_test, y_pred_ridge_unscaled))

# Ridge con escalado
pipeline_ridge = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', Ridge(alpha=1.0))
])
pipeline_ridge.fit(X_train, y_train)
y_pred_ridge_scaled = pipeline_ridge.predict(X_test)
rmse_ridge_scaled = np.sqrt(mean_squared_error(y_test, y_pred_ridge_scaled))

print(f"Ridge sin escalar - RMSE: ${rmse_ridge_unscaled:,.2f}")
print(f"Ridge con escalado - RMSE: ${rmse_ridge_scaled:,.2f}")
print(f"Mejora por escalado: ${rmse_ridge_unscaled - rmse_ridge_scaled:,.2f}")
print("[CONCLUSION] En modelos regularizados, el escalado es CRUCIAL.")

## Experimento 4: Arbol de Decision y Overfitting (controlando `max_depth`)

Los arboles NO necesitan escalado. El hiperparametro `max_depth` controla la complejidad.

In [ ]:
print("\n" + "="*80)
print("EXPERIMENTO 4: ARBOL DE DECISION Y CONTROL DE OVERFITTING")
print("="*80)

# Caso A: Sin limite (overfitting)
tree_overfit = DecisionTreeRegressor(max_depth=None, random_state=42)
tree_overfit.fit(X_train, y_train)
r2_train_over = r2_score(y_train, tree_overfit.predict(X_train))
r2_test_over = r2_score(y_test, tree_overfit.predict(X_test))

print("[CASO A] max_depth = None (SIN LIMITE)")
print(f"   R2 entrenamiento: {r2_train_over:.4f} (casi perfecto)")
print(f"   R2 prueba: {r2_test_over:.4f} (mucho peor)")
print(f"   Diferencia: {r2_train_over - r2_test_over:.4f} -> OVERFITTING")

# Caso B: Con limite (max_depth=5)
tree_pruned = DecisionTreeRegressor(max_depth=5, random_state=42)
tree_pruned.fit(X_train, y_train)
r2_train_prun = r2_score(y_train, tree_pruned.predict(X_train))
r2_test_prun = r2_score(y_test, tree_pruned.predict(X_test))

print("\n[CASO B] max_depth = 5 (CON LIMITE)")
print(f"   R2 entrenamiento: {r2_train_prun:.4f}")
print(f"   R2 prueba: {r2_test_prun:.4f}")
print(f"   Diferencia: {r2_train_prun - r2_test_prun:.4f} -> Overfitting controlado")
print("[CONCLUSION] max_depth es un freno contra el overfitting.")

## Experimento 5: Random Forest (Ensemble de arboles)

Random Forest promedia muchos arboles, reduce la varianza y no necesita escalado.

In [ ]:
print("\n" + "="*80)
print("EXPERIMENTO 5: RANDOM FOREST (100 arboles)")
print("="*80)

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)
mae_rf = mean_absolute_error(y_test, y_pred_rf)

print(f"RMSE: ${rmse_rf:,.2f}")
print(f"R2: {r2_rf:.4f}")
print(f"MAE: ${mae_rf:,.2f}")

# Comparar con la mejor regresion lineal (sin escalar, que da igual)
print(f"\nMejora vs Regresion Lineal (RMSE): ${rmse_lr_unscaled - rmse_rf:,.2f}")
print("[CONCLUSION] Random Forest captura relaciones no lineales y supera a la regresion lineal.")

## Experimento 6: Cross-Validation (K-Fold) para evaluacion robusta

In [ ]:
print("\n" + "="*80)
print("EXPERIMENTO 6: CROSS-VALIDATION (K-Fold=5) en Random Forest")
print("="*80)

scores_neg_mse = cross_val_score(rf, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
scores_rmse = np.sqrt(-scores_neg_mse)

print(f"RMSE promedio en 5 pliegues: ${scores_rmse.mean():,.2f}")
print(f"Desviacion estandar: ${scores_rmse.std():,.2f}")
print(f"RMSE del split simple: ${rmse_rf:,.2f}")
print("[CONCLUSION] Cross-Validation da una estimacion mas estable y realista.")

## Experimento 7: Feature Engineering (crear nuevas variables)

In [ ]:
print("\n" + "="*80)
print("EXPERIMENTO 7: FEATURE ENGINEERING (rooms_per_household)")
print("="*80)

df_encoded['rooms_per_household'] = df_encoded['total_rooms'] / df_encoded['households']

X_fe = df_encoded.drop('median_house_value', axis=1)
X_train_fe, X_test_fe, y_train_fe, y_test_fe = train_test_split(X_fe, y, test_size=0.2, random_state=42)

rf_fe = RandomForestRegressor(n_estimators=100, random_state=42)
rf_fe.fit(X_train_fe, y_train_fe)
y_pred_rf_fe = rf_fe.predict(X_test_fe)

rmse_rf_fe = np.sqrt(mean_squared_error(y_test_fe, y_pred_rf_fe))
r2_rf_fe = r2_score(y_test_fe, y_pred_rf_fe)

print(f"RMSE con nueva feature: ${rmse_rf_fe:,.2f}")
print(f"R2 con nueva feature: {r2_rf_fe:.4f}")
print(f"Mejora vs modelo sin feature: ${rmse_rf - rmse_rf_fe:,.2f}")
print("[CONCLUSION] Un buen Feature Engineering (incluso sugerido por IA) reduce el error.")

## Resumen final (Takeaways para la clase)

1. **Escalado:** NO es necesario para regresion lineal ordinaria, pero SÍ es crucial para modelos regularizados (Ridge, Lasso) y basados en distancias.
2. **Overfitting:** Los arboles sin podar (`max_depth=None`) memorizan los datos. Controlar la profundidad o usar Random Forest lo evita.
3. **Random Forest:** Ensemble de arboles que reduce varianza y captura no linealidades. Suele superar a la regresion lineal.
4. **Metricas:** RMSE castiga outliers, MAE es robusto, R2 mide el % de variabilidad explicada.
5. **Cross-Validation:** Proporciona una evaluacion mas fiable que un unico split.

## Tarea (TP Unidad 2)

Aplicar lo aprendido en el dataset **Insurance Cost**.
- Hacer One-Hot a `smoker` y `region`.
- Probar Regresion Lineal (con y sin escalado) y Ridge (con y sin escalado).
- Probar Random Forest y comparar RMSE y R2.
- Entregar conclusiones sobre el impacto del escalado en cada modelo.